# ⚛️ Módulo 5: Dinámica Molecular
## Actividad 5.7: Análisis de Trayectorias – RMSD, RMSF y Energías

<div align="center">
  
**Universidad de Caldas - Departamento de Química**  
*Introducción a la Química Computacional (173G7G)*  
**Profesor:** José Mauricio Rodas Rodríguez

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maurorodas/Quimica_computacional_173G7G/blob/main/modulo_05_dinamica_molecular/07_analisis_trayectorias.ipynb)

</div>

---

## 🎯 Objetivos de Aprendizaje

Al finalizar esta actividad, serás capaz de:
- Calcular e interpretar el RMSD (desviación cuadrática media) de una proteína durante la simulación
- Calcular el RMSF (fluctuación cuadrática media) por residuo para identificar regiones flexibles
- Analizar el radio de giro (Rg) como medida de compactación de la proteína
- Calcular y graficar el número de puentes de hidrógeno a lo largo de la trayectoria
- Usar MDAnalysis y herramientas de GROMACS para el análisis post-simulación
- Interpretar los resultados en el contexto biológico

---

## 1. Instalación de Dependencias

In [ ]:
!pip install MDAnalysis numpy matplotlib scipy pandas seaborn
# Para análisis con GROMACS usar los comandos gmx directamente

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from scipy.signal import savgol_filter

print("Bibliotecas importadas correctamente")
try:
    import MDAnalysis as mda
    from MDAnalysis.analysis import rms, align
    print(f"MDAnalysis versión: {mda.__version__}")
except ImportError:
    print("MDAnalysis no disponible. Se usarán datos simulados para las demostraciones.")

## 2. RMSD – Desviación Cuadrática Media de Posición

El **RMSD** (Root Mean Square Deviation) mide cuánto se ha desplazado la estructura respecto a una referencia (normalmente la estructura inicial o la estructura cristalina):

$$\text{RMSD}(t) = \sqrt{\frac{1}{N} \sum_{i=1}^{N} m_i \left|\mathbf{r}_i(t) - \mathbf{r}_i^{ref}\right|^2}$$

donde $N$ es el número de átomos, $m_i$ son los pesos (masas) y $\mathbf{r}_i^{ref}$ son las posiciones de referencia.

**Interpretación:**
- RMSD < 2 Å: estructura muy estable, cerca de la referencia
- RMSD 2-4 Å: desviación moderada, normal en proteínas pequeñas
- RMSD > 4 Å: cambio conformacional significativo
- Plateau en el RMSD indica que el sistema ha alcanzado equilibrio

In [ ]:
def calcular_rmsd(posiciones_traj, posiciones_ref, atomos=None):
    """
    Calcula el RMSD de cada frame respecto a una referencia.
    
    Args:
        posiciones_traj: array (n_frames, n_atomos, 3)
        posiciones_ref: array (n_atomos, 3)
        atomos: índices de átomos a incluir (None = todos)
    
    Returns:
        Array de RMSD en Å para cada frame
    """
    if atomos is not None:
        traj = posiciones_traj[:, atomos, :]
        ref  = posiciones_ref[atomos, :]
    else:
        traj = posiciones_traj
        ref  = posiciones_ref
    
    diff = traj - ref[np.newaxis, :, :]
    rmsd = np.sqrt(np.mean(np.sum(diff**2, axis=-1), axis=-1))
    return rmsd

# Generar trayectoria simulada de lisozima (129 residuos, ~1010 átomos pesados Cα)
np.random.seed(42)
n_frames = 1000  # 10 ns con guardado cada 10 ps
n_atomos = 129   # Cα de lisozima

# Posición de referencia aleatoria
pos_ref = np.random.randn(n_atomos, 3) * 10

# Trayectoria: difusión + ruido
traj = np.zeros((n_frames, n_atomos, 3))
traj[0] = pos_ref.copy()
deriva = np.random.randn(3) * 0.01  # deriva lenta del centro de masa
for i in range(1, n_frames):
    # Movimiento de la proteína
    ruido = np.random.randn(n_atomos, 3) * 0.05
    # RMSD que satura alrededor de 2 Å
    factor = 1 - np.exp(-i / 200)
    traj[i] = pos_ref + factor * np.random.randn(n_atomos, 3) * 1.8 + ruido

t_ns = np.linspace(0, 10, n_frames)  # tiempo en ns
rmsd_backbone = calcular_rmsd(traj, pos_ref)

plt.figure(figsize=(10, 5))
plt.plot(t_ns, rmsd_backbone, 'b-', linewidth=0.8, alpha=0.7, label='RMSD bruto')
# Suavizar para visualización
rmsd_suave = savgol_filter(rmsd_backbone, window_length=51, polyorder=3)
plt.plot(t_ns, rmsd_suave, 'r-', linewidth=2, label='RMSD suavizado')
plt.axhline(2.0, color='g', linestyle='--', alpha=0.7, label='2 Å (referencia)')
plt.xlabel('Tiempo (ns)', fontsize=12)
plt.ylabel('RMSD (Å)', fontsize=12)
plt.title('RMSD de Cα - Lisozima en Agua', fontsize=13)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('rmsd_lisozima.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"RMSD medio (últimos 5 ns): {rmsd_backbone[500:].mean():.2f} ± {rmsd_backbone[500:].std():.2f} Å")

## 3. RMSF – Fluctuación Cuadrática Media por Residuo

El **RMSF** (Root Mean Square Fluctuation) mide la flexibilidad de cada residuo respecto a su posición media:

$$\text{RMSF}_i = \sqrt{\langle |\mathbf{r}_i(t) - \langle \mathbf{r}_i \rangle|^2 \rangle_t}$$

**Interpretación:**
- RMSF alto → residuo flexible (loops, terminales N y C)
- RMSF bajo → residuo rígido (núcleo hidrofóbico, hélices, láminas)
- Correlaciona con el **factor B** cristalográfico: $B_i = \frac{8\pi^2}{3} \text{RMSF}_i^2$

In [ ]:
def calcular_rmsf(posiciones_traj, primer_frame=0):
    """
    Calcula el RMSF por átomo sobre la trayectoria.
    
    Args:
        posiciones_traj: array (n_frames, n_atomos, 3)
        primer_frame: inicio del análisis (descartar equilibración)
    
    Returns:
        Array RMSF (n_atomos,) en Å
    """
    traj = posiciones_traj[primer_frame:]
    pos_media = traj.mean(axis=0)  # posición media por átomo
    diff = traj - pos_media[np.newaxis, :, :]
    rmsf = np.sqrt(np.mean(np.sum(diff**2, axis=-1), axis=0))
    return rmsf

# Calcular RMSF a partir de la mitad de la trayectoria (equilibrio)
rmsf_residuos = calcular_rmsf(traj, primer_frame=500)
residuos = np.arange(1, n_atomos + 1)

# Simular factor B experimental (correlacionado con RMSF)
B_factor = (8 * np.pi**2 / 3) * rmsf_residuos**2

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].fill_between(residuos, rmsf_residuos, alpha=0.4, color='steelblue')
axes[0].plot(residuos, rmsf_residuos, 'b-', linewidth=1)
axes[0].axhline(rmsf_residuos.mean(), color='r', linestyle='--',
                label=f'Media = {rmsf_residuos.mean():.2f} Å')
# Destacar residuos muy flexibles
umbral = rmsf_residuos.mean() + rmsf_residuos.std()
axes[0].axhline(umbral, color='orange', linestyle=':', label=f'μ+σ = {umbral:.2f} Å')
axes[0].set_xlabel('Número de residuo', fontsize=12)
axes[0].set_ylabel('RMSF (Å)', fontsize=12)
axes[0].set_title('RMSF por Residuo – Cα de Lisozima', fontsize=13)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Colorear por flexibilidad
sc = axes[1].scatter(residuos, B_factor, c=rmsf_residuos, cmap='RdYlGn_r',
                     s=30, alpha=0.8)
plt.colorbar(sc, ax=axes[1], label='RMSF (Å)')
axes[1].set_xlabel('Número de residuo', fontsize=12)
axes[1].set_ylabel('Factor B calculado (Å²)', fontsize=12)
axes[1].set_title('Factor B calculado a partir del RMSF', fontsize=13)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('rmsf_lisozima.png', dpi=100, bbox_inches='tight')
plt.show()

## 4. Radio de Giro (Rg)

El **radio de giro** mide la compactación global de la proteína:

$$R_g(t) = \sqrt{\frac{\sum_i m_i |\mathbf{r}_i(t) - \mathbf{r}_{cm}(t)|^2}{\sum_i m_i}}$$

Un $R_g$ constante indica que la proteína mantiene su forma global durante la simulación.

In [ ]:
def calcular_radio_giro(posiciones_traj, masas=None):
    """
    Calcula el radio de giro por frame.
    
    Args:
        posiciones_traj: array (n_frames, n_atomos, 3)
        masas: array (n_atomos,) en daltons (None = masas iguales)
    
    Returns:
        Array de Rg (n_frames,) en Å
    """
    n_frames, n_atomos, _ = posiciones_traj.shape
    if masas is None:
        masas = np.ones(n_atomos)
    masas = masas / masas.sum()

    Rg_arr = np.zeros(n_frames)
    for i, pos in enumerate(posiciones_traj):
        cm = np.sum(masas[:, np.newaxis] * pos, axis=0)
        r_cm = pos - cm
        Rg_arr[i] = np.sqrt(np.sum(masas * np.sum(r_cm**2, axis=1)))
    return Rg_arr

Rg = calcular_radio_giro(traj)
Rg_suave = savgol_filter(Rg, 51, 3)

plt.figure(figsize=(10, 4))
plt.plot(t_ns, Rg, 'g-', linewidth=0.7, alpha=0.5, label='Rg bruto')
plt.plot(t_ns, Rg_suave, 'darkgreen', linewidth=2, label='Rg suavizado')
plt.xlabel('Tiempo (ns)', fontsize=12)
plt.ylabel('Radio de giro Rg (Å)', fontsize=12)
plt.title('Radio de Giro – Lisozima en Agua', fontsize=13)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('radio_giro.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Rg medio: {Rg[500:].mean():.2f} ± {Rg[500:].std():.2f} Å")

## 5. Puentes de Hidrógeno

Los puentes de hidrógeno son fundamentales para la estabilidad de la estructura secundaria y terciaria de las proteínas.

**Criterio geométrico:**
- Distancia D···A < 3.5 Å (donador-aceptor)
- Ángulo D-H···A > 120°

In [ ]:
# Análisis de puentes de hidrógeno con MDAnalysis (si está disponible)
try:
    from MDAnalysis.analysis.hydrogenbonds import HydrogenBondAnalysis
    print("MDAnalysis HBA disponible")
    # Ejemplo de uso:
    # u = mda.Universe('topol.tpr', 'traj.xtc')
    # hbonds = HydrogenBondAnalysis(u, 'protein', 'protein')
    # hbonds.run()
    # n_hbonds_time = hbonds.count_by_time()
except (ImportError, Exception):
    print("Usando datos simulados para demostración")

# Datos simulados de puentes de hidrógeno
np.random.seed(0)
n_hb_inicial = 90  # ~90 puentes de H internos en lisozima
n_hb_agua = 200    # puentes de H con agua

n_hb_interno = n_hb_inicial + np.cumsum(np.random.normal(0, 0.5, n_frames)) * 0.1
n_hb_interno = np.clip(n_hb_interno + np.random.normal(0, 3, n_frames), 70, 110)
n_hb_total = n_hb_interno + n_hb_agua + np.random.normal(0, 8, n_frames)

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

axes[0].plot(t_ns, n_hb_interno, 'b-', linewidth=0.8, alpha=0.6)
axes[0].plot(t_ns, savgol_filter(n_hb_interno, 51, 3), 'b-', linewidth=2,
             label=f'HB intramolecular ({n_hb_interno[500:].mean():.0f} media)')
axes[0].set_ylabel('N° puentes de H', fontsize=12)
axes[0].set_title('Puentes de Hidrógeno Intramoleculares', fontsize=13)
axes[0].legend(fontsize=11); axes[0].grid(True, alpha=0.3)

axes[1].plot(t_ns, n_hb_total, 'g-', linewidth=0.8, alpha=0.6)
axes[1].plot(t_ns, savgol_filter(n_hb_total, 51, 3), 'g-', linewidth=2,
             label=f'HB total (interno + agua) ({n_hb_total[500:].mean():.0f} media)')
axes[1].set_xlabel('Tiempo (ns)', fontsize=12)
axes[1].set_ylabel('N° puentes de H', fontsize=12)
axes[1].set_title('Puentes de Hidrógeno Totales (Proteína + Agua)', fontsize=13)
axes[1].legend(fontsize=11); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('puentes_hidrogeno.png', dpi=100, bbox_inches='tight')
plt.show()

## 6. Comandos GROMACS para Análisis Post-Simulación

In [ ]:
comandos_analisis = """
# ============================================================
# ANÁLISIS DE TRAYECTORIA CON GROMACS
# ============================================================

# 1. Corregir periodicidad (centrar proteína)
echo "1 0" | gmx trjconv -s md.tpr -f md.xtc -o md_center.xtc \\
    -center -pbc mol -ur compact

# 2. RMSD de Cα respecto a la estructura inicial
echo "4 4" | gmx rms -s md.tpr -f md_center.xtc \\
    -o rmsd_ca.xvg -tu ns
# Seleccionar: 4 = C-alpha

# 3. RMSF por residuo
echo "4" | gmx rmsf -s md.tpr -f md_center.xtc \\
    -o rmsf_residues.xvg -res

# 4. Radio de giro
echo "1" | gmx gyrate -s md.tpr -f md_center.xtc \\
    -o gyrate.xvg

# 5. Puentes de hidrógeno
echo "1 1" | gmx hbond -s md.tpr -f md_center.xtc \\
    -num hbnum.xvg -dist hbdist.xvg

# 6. Energías potenciales
echo "Potential\n0" | gmx energy -f md.edr -o energy.xvg

# 7. Temperatura y presión
echo "Temperature\nPressure\n0" | gmx energy -f md.edr -o temp_press.xvg

# 8. Densidad
echo "Density\n0" | gmx energy -f md.edr -o density.xvg

# 9. Análisis de estructura secundaria (DSSP)
echo "1" | gmx do_dssp -s md.tpr -f md_center.xtc \\
    -o ss.xpm -sc ss_count.xvg

# 10. Análisis de solvatación / SASA
echo "1" | gmx sasa -s md.tpr -f md_center.xtc \\
    -o sasa.xvg -or sasa_residues.xvg
"""
print(comandos_analisis)
Path('scripts').mkdir(exist_ok=True)
with open('scripts/analisis.sh', 'w') as f:
    f.write("#!/bin/bash\n" + comandos_analisis)
print("Script guardado en: scripts/analisis.sh")

## 7. Análisis con MDAnalysis

In [ ]:
# Ejemplo completo de análisis con MDAnalysis
codigo_mdanalysis = """
import MDAnalysis as mda
from MDAnalysis.analysis import rms, align, contacts
import numpy as np
import matplotlib.pyplot as plt

# Cargar trayectoria
u = mda.Universe('md.tpr', 'md_center.xtc')
proteina = u.select_atoms('protein')
calfa = u.select_atoms('name CA')

# RMSD
R = rms.RMSD(calfa, ref_frame=0)
R.run()
t_ns = R.results.rmsd[:, 1] / 1000  # ps a ns
rmsd = R.results.rmsd[:, 2]  # Å

# RMSF
# Alinear primero la trayectoria
aligner = align.AlignTraj(u, u, select='name CA', in_memory=True)
aligner.run()
F = rms.RMSF(calfa)
F.run()
rmsf = F.results.rmsf  # Å por residuo

# Graficar
fig, axes = plt.subplots(2, 1, figsize=(12, 8))
axes[0].plot(t_ns, rmsd)
axes[0].set(xlabel='Tiempo (ns)', ylabel='RMSD (Å)', title='RMSD Cα')
axes[1].plot(calfa.resnums, rmsf)
axes[1].set(xlabel='Residuo', ylabel='RMSF (Å)', title='RMSF por residuo')
plt.tight_layout()
plt.savefig('analisis_mdanalysis.png', dpi=100)
plt.show()
"""

try:
    import MDAnalysis as mda
    print("MDAnalysis disponible. Puedes ejecutar el código de referencia si tienes una trayectoria.")
except ImportError:
    print("MDAnalysis no instalado.")

print("\nCódigo de referencia para MDAnalysis:")
print(codigo_mdanalysis)

## 8. Panel de Análisis Completo

In [ ]:
# Panel resumen de análisis de trayectoria
fig = plt.figure(figsize=(16, 12))

# RMSD
ax1 = fig.add_subplot(3, 2, 1)
ax1.plot(t_ns, rmsd_backbone, 'b-', linewidth=0.6, alpha=0.5)
ax1.plot(t_ns, savgol_filter(rmsd_backbone, 51, 3), 'b-', linewidth=2)
ax1.set(xlabel='Tiempo (ns)', ylabel='RMSD (Å)', title='RMSD Cα')
ax1.grid(True, alpha=0.3)

# RMSF
ax2 = fig.add_subplot(3, 2, 2)
ax2.fill_between(residuos, rmsf_residuos, alpha=0.4, color='green')
ax2.plot(residuos, rmsf_residuos, 'g-', linewidth=1)
ax2.set(xlabel='Residuo', ylabel='RMSF (Å)', title='RMSF por residuo')
ax2.grid(True, alpha=0.3)

# Radio de giro
ax3 = fig.add_subplot(3, 2, 3)
ax3.plot(t_ns, Rg, 'purple', linewidth=0.6, alpha=0.5)
ax3.plot(t_ns, savgol_filter(Rg, 51, 3), 'purple', linewidth=2)
ax3.set(xlabel='Tiempo (ns)', ylabel='Rg (Å)', title='Radio de Giro')
ax3.grid(True, alpha=0.3)

# Puentes de H
ax4 = fig.add_subplot(3, 2, 4)
ax4.plot(t_ns, n_hb_interno, 'r-', linewidth=0.6, alpha=0.5)
ax4.plot(t_ns, savgol_filter(n_hb_interno, 51, 3), 'r-', linewidth=2)
ax4.set(xlabel='Tiempo (ns)', ylabel='N° HB', title='Puentes de Hidrógeno Intramoleculares')
ax4.grid(True, alpha=0.3)

# Distribución RMSD
ax5 = fig.add_subplot(3, 2, 5)
ax5.hist(rmsd_backbone[500:], bins=40, color='steelblue', edgecolor='white', alpha=0.8)
ax5.set(xlabel='RMSD (Å)', ylabel='Frecuencia', title='Distribución de RMSD (eq.)')
ax5.grid(True, alpha=0.3)

# RMSD vs Rg (mapa de conformaciones)
ax6 = fig.add_subplot(3, 2, 6)
sc = ax6.scatter(rmsd_backbone[::5], Rg[::5], c=t_ns[::5], cmap='viridis',
                 s=10, alpha=0.6)
plt.colorbar(sc, ax=ax6, label='Tiempo (ns)')
ax6.set(xlabel='RMSD (Å)', ylabel='Rg (Å)', title='Espacio de Conformaciones')
ax6.grid(True, alpha=0.3)

plt.suptitle('Panel de Análisis de Trayectoria – Lisozima (simulación)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('panel_analisis.png', dpi=100, bbox_inches='tight')
plt.show()

## 9. Ejercicios

### Ejercicio 1 (Básico)
Calcula el RMSD de una trayectoria donde la proteína sufre un cambio conformacional a los 5 ns (por ejemplo, aumenta el RMSD de 2 a 5 Å). Genera una gráfica e identifica el momento del cambio.

### Ejercicio 2 (Intermedio)
Implementa el cálculo del área de superficie accesible al solvente (SASA) para cada frame de la trayectoria usando el algoritmo de esferas rodantes. Compara el SASA de la proteína en simulación con el de la estructura cristalina.

### Ejercicio 3 (Avanzado)
Analiza una trayectoria real de GROMACS usando `gmx rms`, `gmx rmsf` y `gmx gyrate`. Crea un script Python que lea los archivos `.xvg` generados por GROMACS y produzca el panel de análisis completo mostrado en esta actividad.

In [ ]:
# Función para leer archivos .xvg de GROMACS
def leer_xvg(filename):
    """
    Lee un archivo .xvg de GROMACS y devuelve los datos como array.
    
    Args:
        filename: ruta al archivo .xvg
    
    Returns:
        numpy array con los datos (ignora líneas de comentario)
    """
    datos = []
    with open(filename, 'r') as f:
        for linea in f:
            linea = linea.strip()
            if linea and not linea.startswith(('#', '@')):
                try:
                    valores = [float(x) for x in linea.split()]
                    datos.append(valores)
                except ValueError:
                    pass
    return np.array(datos)

# Demostración con datos simulados de RMSD
# En uso real: datos = leer_xvg('rmsd_ca.xvg')
print("Función leer_xvg definida correctamente.")
print("Uso: datos = leer_xvg('rmsd_ca.xvg')")
print("     t_ns = datos[:, 0]  # columna tiempo")
print("     rmsd = datos[:, 1]  # columna RMSD")

## 10. Recursos Adicionales

- **MDAnalysis:**
  - [MDAnalysis Documentación](https://www.mdanalysis.org/)
  - [MDAnalysis User Guide](https://userguide.mdanalysis.org/)
  - [MDAnalysis Tutorial](https://www.mdanalysis.org/MDAnalysisTutorial/)

- **GROMACS análisis:**
  - [GROMACS Analysis Tools](https://manual.gromacs.org/documentation/current/reference-manual/analysis.html)
  - [Tutorial análisis GROMACS](http://www.mdtutorials.com/gmx/lysozyme/06_analysis.html)

- **Visualización:**
  - [VMD](https://www.ks.uiuc.edu/Research/vmd/) — visualización de trayectorias
  - [PyMOL](https://pymol.org/) — análisis estructural y visualización

---

## ✅ Verificación de Aprendizaje

Al finalizar esta actividad deberías ser capaz de:

- ✅ Calcular e interpretar el RMSD de una proteína durante la simulación
- ✅ Calcular el RMSF por residuo e identificar regiones flexibles
- ✅ Analizar el radio de giro como medida de compactación global
- ✅ Calcular y graficar los puentes de hidrógeno a lo largo de la trayectoria
- ✅ Usar MDAnalysis y herramientas GROMACS para el análisis post-simulación
- ✅ Interpretar los resultados en el contexto biológico

---

<div align="center">

## 🎉 ¡Felicitaciones!

Has completado la **Actividad 5.7: Análisis de Trayectorias**

[![Anterior](https://img.shields.io/badge/⬅️_Actividad_5.6-Simulación_de_Proteínas-blue.svg)](06_simulacion_proteinas.ipynb)
[![Siguiente](https://img.shields.io/badge/Actividad_5.8_➡️-Práctica_GROMACS_OpenMM-green.svg)](08_practica_gromacs_openmm.ipynb)

---

📚 **[Volver al Módulo 5](README.md)** | 🏠 **[Inicio del Curso](../README.md)**

---

**Universidad de Caldas - Departamento de Química**  
*Química Computacional 173G7G*

</div>